In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from source.random_walk import generate_bat_flight

from source.attractor_networks.single_bump import ToroidZhang1996

# from source.attractor_networks.single_bump import ToroidZhang1996

import tqdm
import os

In [ ]:
## Simulation parameters:
n_min_plots = 10
dt = 0.5e-3
n = 64

net = ToroidZhang1996(n=128, dt=dt)
net.warm_up()
ncols = np.int64(np.ceil(np.sqrt(n_min_plots)))
nrows = np.int64(np.ceil(n_min_plots / ncols))
n_plots = nrows * ncols
fig, ax = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
ax = ax.flatten()
phi = 0
omega = 5
omega_max = 50
n_steps = int(2 * np.pi / omega / dt)
records = np.linspace(0, n_steps - 1, n_plots, dtype=int)
plot_counter = 0
time = 0


for step_iter in range(n_steps):
    if step_iter == records[plot_counter]:
        ax[plot_counter].imshow(net.s)
        ax[plot_counter].set_title(f"$\\phi/2\\pi$={phi/(2*np.pi):.2f}")
        plot_counter += 1

    if step_iter == n_steps // 2:
        net.step(omega_max, 0)
        ax[plot_counter].imshow(net.s)
        ax[plot_counter].set_title(f"$\\phi/2\\pi$={phi/(2*np.pi):.2f} (Sudden turn)")
        plot_counter += 1
    else:
        net.step(omega, 0)
    time += dt
    phi += omega * dt

In [ ]:
T = 1000
net = ToroidZhang1996(n=n, dt=dt)
net.warm_up()
plt.imshow(net.s)
save_dir = "simulation_data"

In [ ]:
bat_flight = generate_bat_flight(
    T=T,
    dt=dt,
    boxsize=5.0,
    speed_mean=2.0,
    speed_std=0.4,
    speed_correlation_time=0.5,
    max_speed=5.0,
    turning_std=0.5,
    turning_correlation_time=0.2,
    max_angular_velocity=2.0,
    wall_repulsion_strength=1.5,
    wall_turn_max=2.0,
    turn_clearance=1.0,
    slow_clearance=1.5,
    wall_slow_speed=0.4,
    rng=np.random.default_rng(42),
)
recorded_cells = [
    (i, j)
    for i in np.random.randint(0, n - 1, size=3)
    for j in np.random.randint(0, n - 1, size=3)
]

n_steps = bat_flight["time"].shape[0]
n_popul_snapshots = 1000
recording = np.zeros((n_steps, len(recorded_cells)))
population_recordings = np.zeros((n_popul_snapshots, n, n))
popul_snapshots_integers = np.linspace(0, n_steps, n_popul_snapshots, dtype=int)
snapshot_iter = 0
for step_iter in tqdm.tqdm(range(n_steps)):
    recording[step_iter] = np.array(
        [net.s[cell_index] for cell_index in recorded_cells]
    )
    net.step(*bat_flight["dir_vel"][step_iter])
    if step_iter == popul_snapshots_integers[snapshot_iter]:
        population_recordings[snapshot_iter] = net.s
        step_iter += 1

os.makedirs(save_dir, exist_ok=True)
np.save(os.path.join(save_dir, "bat_single_bump_cell_recording.npy"), recording)
np.save(os.path.join(save_dir, "bat_single_bump_cell_population.npy"), recording)
np.save(os.path.join(save_dir, "bat_single_bump_cell_position.npy"), bat_flight["pos"])
np.save(os.path.join(save_dir, "bat_single_bump_cell_velocity.npy"), bat_flight["vel"])
np.save(os.path.join(save_dir, "bat_single_bump_cell_direction.npy"), bat_flight["dir"])
np.save(
    os.path.join(save_dir, "bat_single_bump_cell_turn_velocity.npy"),
    bat_flight["dir_vel"],
)
np.save(os.path.join(save_dir, "bat_single_bump_cell_time.npy"), bat_flight["time"])

In [ ]:
bat_flight = {}
recording = np.load(os.path.join(save_dir, "bat_single_bump_cell_recording.npy"))
bat_flight["pos"] = np.load(os.path.join(save_dir, "bat_single_bump_cell_position.npy"))
bat_flight["vel"] = np.load(os.path.join(save_dir, "bat_single_bump_cell_velocity.npy"))
bat_flight["dir"] = np.load(
    os.path.join(save_dir, "bat_single_bump_cell_direction.npy")
)
bat_flight["dir_vel"] = np.load(
    os.path.join(save_dir, "bat_single_bump_cell_turn_velocity.npy")
)
bat_flight["time"] = np.load(os.path.join(save_dir, "bat_single_bump_cell_time.npy"))
n_steps = bat_flight["time"].shape[0]

In [ ]:
counts, edges = np.histogramdd(bat_flight["pos"])

In [ ]:
from source.plot_tools import activity_map

azimuth_activity_map, edges = activity_map(bat_flight["dir"], recording[:, 3], nbins=25)